In [1]:
import sys
from pathlib import Path
import geopandas as gpd
from datetime import datetime, timezone
import pandas as pd
from tqdm import tqdm
import shutil

# Find the repository root.
PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src").is_dir()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_config, photo_paths
from src.prep import (
    download_photo,
    get_image_metadata,
    calculate_ground_extent,
    ground_to_mercator_extent,
    update_tracker,
)

cfg = load_config()

In [2]:
INDEX_PATH = cfg["paths"]["index"]
print("Source index:", INDEX_PATH)
print("County:", cfg["county"])

Source index: C:\Users\eu01251115\LocalProjects\batch_georef\data\main_index_blank.gpkg
County: Carver


In [3]:
# Load the main index.
LAYER = "photos"
index_gdf = gpd.read_file(INDEX_PATH, layer=LAYER)

### Examine and choose available years and projects
The first cell shows all available years for the chosen county. 
The second cell below displays unique combos between flight codes and years. 

In [4]:
# examine all years in the county

county_gdf = index_gdf.loc[index_gdf["county"] == cfg["county"]]

print(cfg["county"], "county flight years:")

for year in sorted(county_gdf["year"].dropna().unique()):
    print(year)

Carver county flight years:
1937
1938
1940
1947
1951
1953
1955
1956
1957
1960
1962
1963
1964
1967
1969
1971


In [5]:
# Examine unique year + flight code combinations for the county
county_gdf = index_gdf.loc[index_gdf["county"] == cfg["county"]]

flights = (
    county_gdf[["year", "flight_code", "county"]]
    .dropna()
    .drop_duplicates()
    .sort_values(["year", "flight_code", "county"])
)

display(flights)

,year,flight_code,county
13511,1937,Carver,Carver
13788,1937,Hennepin,Carver
13773,1937,Scott,Carver
13804,1938,bjm,Carver
13809,1940,Carver,Carver
13886,1940,McLeod,Carver
13894,1940,Scott,Carver
13897,1947,dv,Carver
14059,1951,Carver,Carver
14173,1951,Scott,Carver


### Manually determining ground scale:

- Visit the [MHAPO Air Photo Holdings page](https://www.lib.umn.edu/collections/borchert/aerial-photography-and-remote-sensed-imagery). 
- Select your chosen county to open a PDF with metadata. 
- Compare this to the list above to determine the ground scale for your target year or years. Currently, this tool supports processing multiple years at the same time, but only if they share the same ground scale. 
- **Choose a year or multiple years with the same scale**
- Specify the year(s) and scale for your current batch in config.yml.
- The batch will be named for the county, years, and scale. The photos and the folders they are in will be named for the county and year. 

#### Once you have configured your year(s) and scale in config.yml, restart the kernel and start again from the beginning.


In [6]:
# Select records for the configured county and years.
mask = (
    index_gdf["county"].str.casefold().eq(
        cfg["county"].casefold()
    )
    & index_gdf["year"].isin(cfg["years"])
)

In [7]:
batch_gdf = index_gdf.loc[mask].copy()

In [8]:
TRACKER_PATH = cfg["paths"]["tracker"]
LAYER = "photos"
print("Path for tracker:", TRACKER_PATH)
print("County:", cfg["county"])
print("Years:", cfg["years"])
print("Scale:", cfg["scale"])

Path for tracker: C:\Users\eu01251115\LocalProjects\batch_georef\data\batches\carver_1963_20000_tracker.gpkg
County: Carver
Years: [1963]
Scale: 20000


## Create a tracker table

In [9]:
# validate this set of records
if batch_gdf.empty:
    raise ValueError("No photographs match this batch configuration.")

if (
    batch_gdf["cty_year_id"].isna().any()
    or not batch_gdf["cty_year_id"].is_unique
):
    raise ValueError("Batch contains missing or duplicate photo IDs.")

In [10]:
# check length and confirm CRS
print(f"Selected {len(batch_gdf)} photographs")
print("CRS:", batch_gdf.crs)

Selected 100 photographs
CRS: EPSG:3857


In [11]:
# Assign the configured scale to all selected photographs.
batch_gdf["scale"] = cfg["scale"]

In [12]:
# preview it
display(
    batch_gdf[
        ["cty_year_id", "county", "year", "scale", "url"]
    ].head()
)

,cty_year_id,county,year,scale,url
14371,carver_1963_wm-1dd-3,Carver,1963,20000,https://geo.lib.umn.edu/carver/y1963/wm-1dd-3.jpg
14372,carver_1963_wm-1dd-7,Carver,1963,20000,https://geo.lib.umn.edu/carver/y1963/wm-1dd-7.jpg
14373,carver_1963_wm-1dd-9,Carver,1963,20000,https://geo.lib.umn.edu/carver/y1963/wm-1dd-9.jpg
14374,carver_1963_wm-1dd-11,Carver,1963,20000,https://geo.lib.umn.edu/carver/y1963/wm-1dd-11...
14375,carver_1963_wm-1dd-13,Carver,1963,20000,https://geo.lib.umn.edu/carver/y1963/wm-1dd-13...


In [13]:
batch_gdf.columns

Index(['cty_year_id', 'status', 'county', 'year', 'photo_id', 'ground_width',
       'ground_height', 'wm_width', 'wm_height', 'scale', 'dpi_x', 'dpi_y',
       'width_px', 'height_px', 'worldfile_json', 'url', 'flight_code',
       'source_id', 'date_prepared', 'date_georef', 'gcp_count', 'confidence',
       'error', 'latitude', 'geometry'],
      dtype='str')

In [14]:
if TRACKER_PATH.exists():
    print(f"Using existing batch tracker: {TRACKER_PATH}")

else:
    TRACKER_PATH.parent.mkdir(parents=True, exist_ok=True)

    batch_gdf.to_file(
        TRACKER_PATH,
        layer=LAYER,
        driver="GPKG",
    )

    print(f"Created batch tracker: {TRACKER_PATH}")

Using existing batch tracker: C:\Users\eu01251115\LocalProjects\batch_georef\data\batches\carver_1963_20000_tracker.gpkg


## Download JPEGS and calculate extents

In [15]:
# Load the batch tracker for preparation
tracker_gdf = gpd.read_file(TRACKER_PATH, layer=LAYER)

print(f"Batch: {cfg['batch_name']}")
print(f"Photographs: {len(tracker_gdf)}")

display(tracker_gdf.head())

Batch: carver_1963_20000
Photographs: 100


,cty_year_id,status,county,year,photo_id,ground_width,ground_height,wm_width,wm_height,scale,...,error,latitude,match_count,inlier_count,geometric_score,distribution_quality,match_seconds,georef_status,georef_error,geometry
0,carver_1963_wm-1dd-3,prepared,Carver,1963,wm-1dd-3,4616.873333,4628.726667,6525.345815,6542.098949,20000,...,None,44.965754,512.0,223.0,NaN,0.614325,10.435253,success,None,POINT (-10440436.744 5616131.764)
1,carver_1963_wm-1dd-7,prepared,Carver,1963,wm-1dd-7,4650.740000,4615.180000,6573.197246,6522.937955,20000,...,None,44.965626,512.0,300.0,NaN,0.619352,10.280280,success,None,POINT (-10449562.493 5616111.626)
2,carver_1963_wm-1dd-9,prepared,Carver,1963,wm-1dd-9,4643.966667,4628.726667,6563.614207,6542.074544,20000,...,None,44.965540,512.0,58.0,NaN,0.632380,9.954400,success,None,POINT (-10454125.256 5616098.095)
3,carver_1963_wm-1dd-11,prepared,Carver,1963,wm-1dd-11,4643.966667,4628.726667,6563.602765,6542.063139,20000,...,None,44.965440,512.0,200.0,NaN,0.621493,9.783661,success,None,POINT (-10458688.02 5616082.361)
4,carver_1963_wm-1dd-13,prepared,Carver,1963,wm-1dd-13,4637.193333,4621.953333,6554.016459,6532.476876,20000,...,None,44.965325,146.0,8.0,NaN,0.775083,9.860708,success,None,POINT (-10463250.783 5616064.268)


In [16]:
# Download original JPEGs.
download_results = []

for _, photo in tqdm(
    tracker_gdf.iterrows(),
    total=len(tracker_gdf),
    desc="Downloading photographs",
):
    photo_id = str(photo["cty_year_id"])
    jpeg_path = photo_paths(photo, cfg)["jpeg"]

    try:
        download_photo(photo["url"], jpeg_path)

        download_results.append({
            "cty_year_id": photo_id,
            "jpeg_path": str(jpeg_path),
            "error": None,
        })

    except Exception as exc:
        print(f"FAILED: {photo_id}: {exc}")

        download_results.append({
            "cty_year_id": photo_id,
            "jpeg_path": str(jpeg_path),
            "error": str(exc),
        })

download_df = pd.DataFrame(download_results)

print(f"Successful downloads: {download_df['error'].isna().sum()}")
print(f"Failed downloads: {download_df['error'].notna().sum()}")

display(download_df.head())

Successful downloads: 100
Failed downloads: 0


,cty_year_id,jpeg_path,error
0,carver_1963_wm-1dd-3,C:\Users\eu01251115\LocalProjects\batch_georef...,None
1,carver_1963_wm-1dd-7,C:\Users\eu01251115\LocalProjects\batch_georef...,None
2,carver_1963_wm-1dd-9,C:\Users\eu01251115\LocalProjects\batch_georef...,None
3,carver_1963_wm-1dd-11,C:\Users\eu01251115\LocalProjects\batch_georef...,None
4,carver_1963_wm-1dd-13,C:\Users\eu01251115\LocalProjects\batch_georef...,None


In [20]:
# Collect preparation results for each photo.
results = []

for _, photo in tqdm(
    tracker_gdf.iterrows(),
    total=len(tracker_gdf),
    desc="Calculating extents",
):
    photo_id = photo["cty_year_id"]
    image_path = photo_paths(photo, cfg)["jpeg"]

    result = {
        "cty_year_id": photo_id,
        "width_px": None,
        "height_px": None,
        "dpi_x": None,
        "dpi_y": None,
        "ground_width": None,
        "ground_height": None,
        "wm_width": None,
        "wm_height": None,
        "date_prepared": None,
        "error": None,
    }

    try:
        # Read metadata from the existing JPEG.
        metadata = get_image_metadata(image_path)
        result.update(metadata)

        # Calculate ground dimensions in meters.
        result["ground_width"] = calculate_ground_extent(
            pixels=metadata["width_px"],
            dpi=metadata["dpi_x"],
            scale=photo["scale"],
        )

        result["ground_height"] = calculate_ground_extent(
            pixels=metadata["height_px"],
            dpi=metadata["dpi_y"],
            scale=photo["scale"],
        )

        # Convert dimensions to Web Mercator.
        result["wm_width"] = ground_to_mercator_extent(
            ground_extent_m=result["ground_width"],
            latitude=photo["latitude"],
        )

        result["wm_height"] = ground_to_mercator_extent(
            ground_extent_m=result["ground_height"],
            latitude=photo["latitude"],
        )

        # Mark preparation complete only after successful calculations.
        result["date_prepared"] = datetime.now(
            timezone.utc
        ).date().isoformat()

    except Exception as e:
        result["error"] = str(e)
        print(f"FAILED: {photo_id}: {e}")

    results.append(result)

# Convert results to a DataFrame for inspection.
results_df = pd.DataFrame(results)

print(f"Successful: {results_df['error'].isna().sum()}")
print(f"Failed: {results_df['error'].notna().sum()}")

display(results_df.head())

Calculating extents: 100%|██████████| 100/100 [00:00<00:00, 1731.20it/s]

Successful: 100
Failed: 0


,cty_year_id,width_px,height_px,dpi_x,dpi_y,ground_width,ground_height,wm_width,wm_height,date_prepared,error
0,carver_1963_wm-1dd-3,5453,5467,600,600,4616.873333,4628.726667,6525.345815,6542.098949,2026-11-25,None
1,carver_1963_wm-1dd-7,5493,5451,600,600,4650.740000,4615.180000,6573.197246,6522.937955,2026-11-25,None
2,carver_1963_wm-1dd-9,5485,5467,600,600,4643.966667,4628.726667,6563.614207,6542.074544,2026-11-25,None
3,carver_1963_wm-1dd-11,5485,5467,600,600,4643.966667,4628.726667,6563.602765,6542.063139,2026-11-25,None
4,carver_1963_wm-1dd-13,5477,5459,600,600,4637.193333,4621.953333,6554.016459,6532.476876,2026-11-25,None


In [21]:
existing_tracker = gpd.read_file(TRACKER_PATH, layer=LAYER)

display(pd.DataFrame({
    "tracker_dtype": existing_tracker.dtypes.astype(str),
    "results_dtype": results_df.dtypes.astype(str),
}).loc[results_df.columns])

,tracker_dtype,results_dtype
cty_year_id,str,str
width_px,float64,int64
height_px,float64,int64
dpi_x,float64,int64
dpi_y,float64,int64
ground_width,float64,float64
ground_height,float64,float64
wm_width,float64,float64
wm_height,float64,float64
date_prepared,str,str


In [22]:
# Update the tracker table
updated_count = update_tracker(
    TRACKER_PATH,
    results_df,
    layer=LAYER,
)

print(f"Updated {updated_count} records.")

Updated 100 records.


### Final load & preview

In [23]:
tracker_gdf = gpd.read_file(TRACKER_PATH, layer=LAYER)

tracker_gdf

,cty_year_id,status,county,year,photo_id,ground_width,ground_height,wm_width,wm_height,scale,...,error,latitude,match_count,inlier_count,geometric_score,distribution_quality,match_seconds,georef_status,georef_error,geometry
0,carver_1963_wm-1dd-3,prepared,Carver,1963,wm-1dd-3,4616.873333,4628.726667,6525.345815,6542.098949,20000,...,None,44.965754,512.0,223.0,NaN,0.614325,10.435253,success,None,POINT (-10440436.744 5616131.764)
1,carver_1963_wm-1dd-7,prepared,Carver,1963,wm-1dd-7,4650.740000,4615.180000,6573.197246,6522.937955,20000,...,None,44.965626,512.0,300.0,NaN,0.619352,10.280280,success,None,POINT (-10449562.493 5616111.626)
2,carver_1963_wm-1dd-9,prepared,Carver,1963,wm-1dd-9,4643.966667,4628.726667,6563.614207,6542.074544,20000,...,None,44.965540,512.0,58.0,NaN,0.632380,9.954400,success,None,POINT (-10454125.256 5616098.095)
3,carver_1963_wm-1dd-11,prepared,Carver,1963,wm-1dd-11,4643.966667,4628.726667,6563.602765,6542.063139,20000,...,None,44.965440,512.0,200.0,NaN,0.621493,9.783661,success,None,POINT (-10458688.02 5616082.361)
4,carver_1963_wm-1dd-13,prepared,Carver,1963,wm-1dd-13,4637.193333,4621.953333,6554.016459,6532.476876,20000,...,None,44.965325,146.0,8.0,NaN,0.775083,9.860708,success,None,POINT (-10463250.783 5616064.268)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,carver_1963_wm-1dd-111,prepared,Carver,1963,wm-1dd-111,4616.873333,4635.500000,6495.562927,6521.769122,20000,...,None,44.702124,NaN,NaN,NaN,NaN,NaN,NaN,None,POINT (-10428717.028 5574748.276)
96,carver_1963_wm-1dd-113,prepared,Carver,1963,wm-1dd-113,4637.193333,4635.500000,6524.206835,6521.824434,20000,...,None,44.702615,NaN,NaN,NaN,NaN,NaN,NaN,None,POINT (-10424133.337 5574825.175)
97,carver_1963_wm-1dd-228,prepared,Carver,1963,wm-1dd-228,4645.660000,4615.180000,6532.806900,6489.945401,20000,...,None,44.673257,NaN,NaN,NaN,NaN,NaN,NaN,None,POINT (-10435676.055 5570228.328)
98,carver_1963_wm-1dd-230,prepared,Carver,1963,wm-1dd-230,4638.886667,4608.406667,6523.289439,6480.427892,20000,...,None,44.673322,NaN,NaN,NaN,NaN,NaN,NaN,None,POINT (-10431122.308 5570238.503)
